# Comparación de modelos fundacionales de Time Series. Cambios respecto a la prueba 2:
Implementación de los modelos fundacionales que no estan originalmente en timecopilot
Los pruebo

**1. Instalo paquetes**

In [0]:
%pip install --upgrade black fsspec
%pip install --no-deps "momentfm==0.1.4"
%pip install --no-deps "git+https://github.com/hitachi-ais/citras-fm.git@v3.0.0"
%pip install --no-deps "tabicl[forecast]==2.0.3"

In [0]:
#instalo las cositas
%pip install --no-deps -e /Workspace/Users/anamaria.afonso@practicas.repsol.com/timecopilot_pruebas
# dependencias de timecopilot que no se instalan por usar --no-deps
%pip install "statsforecast>=2.0.2" "catboost>=1.2.10" fire "logfire>=4.11.0" "openai>=1.99.7" "opentelemetry-instrumentation>=0.58b0" "peft>=0.20.0,<1" "plotly>=6.3.1" "pydantic-ai>=0.7.0" "xgboost>=3.2.0"
%pip install "pydantic-ai-slim[google]"
%pip install tsfeatures
%pip install foundationforecast
%pip install mlforecast
%pip install neuralforecast
%pip install prophet

In [0]:
%pip install "transformers==4.57.6"

In [0]:
dbutils.library.restartPython()

**2. Importo mis librerias**

In [0]:
#importo librerias necesarias
import timecopilot
import os
import sys
import pandas as pd
import nest_asyncio
#from timecopilot import TimeCopilot
nest_asyncio.apply()
from timecopilot import TimeCopilotForecaster
#importo los modelos que quiero usar
#timecopilot
from timecopilot.models.foundation.chronos import Chronos
from timecopilot.models.foundation.tirex import TiRex
from timecopilot.models.foundation import Moirai
#agregados por mi, confirmado que ya funcionan
import sys
sys.path.insert(0, "/Workspace/Users/anamaria.afonso@practicas.repsol.com/Time-MoE")
from timecopilot.models.foundation.time_moe import TimeMoE 
LAG_LLAMA_ROOT = (
    "/Workspace/Users/anamaria.afonso@practicas.repsol.com/lag-llama"
)
if LAG_LLAMA_ROOT in sys.path:
    sys.path.remove(LAG_LLAMA_ROOT)
sys.path.insert(0, LAG_LLAMA_ROOT)
from timecopilot.models.foundation.lag_llama import LagLlama
from timecopilot.models.foundation.moment import MOMENT
from timecopilot.models.foundation.citras_fm import CITRASFM
#importo metodos de evaluacion
from functools import partial
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import (
    # Errores absolutos/cuadráticos
    mae,
    mse,
    rmse,
    # Errores porcentuales
    mape,
    smape,
    wape,
    # Errores escalados
    mase,
    msse,
    rmsse,
    # Sesgo
    bias,
    cfe,
    # Métricas de inventario
    pis,
    spis,
    # Probabilísticas
    coverage,
    calibration,
    winkler_score,
    scaled_crps,
)

from functools import partial

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from plotly.colors import qualitative

In [0]:
#importacion a prueba
from timecopilot.models.foundation.tabicl_v2 import TabICLv2

**3. Importo mis APIs**

In [0]:
import os

os.environ["HF_TOKEN"] = dbutils.secrets.get(
    scope="timecopilot",
    key="hf-token",
)
os.environ["NIXTLA_API_KEY"] = dbutils.secrets.get(
    scope="timecopilot",
    key="nixtila_api_key",
)
os.environ["TABPFN_TOKEN"] = dbutils.secrets.get(
    scope="timecopilot",
    key="tabpfn_token",
)

**4. Importo mi DF**

In [0]:
df =pd.read_csv(
    "https://timecopilot.s3.amazonaws.com/public/data/events_pageviews.csv",
    parse_dates=["ds"],
)
df.head()
TimeCopilotForecaster.plot(df)

**5. Creo mi Time Copilot Forecaster**

In [0]:
tcf = TimeCopilotForecaster(
    models=[
        TimeMoE(
            repo_id="Maple728/TimeMoE-50M",
            alias="TimeMoE-50M",
            context_length=512,
            seasonality=12,
            calibration_windows=5,
            calibration_step_size=1,
        ),
        Chronos(
            repo_id="amazon/chronos-bolt-tiny",
            alias="Chronos-bolt-tiny",
        ),
        TiRex(repo_id="NX-AI/TiRex", alias="TiRex"),
        Moirai(repo_id="Salesforce/moirai-1.0-R-small", alias="Moirai"),
        LagLlama(alias="Lag-Llama", num_samples=100),
        MOMENT(alias="MOMENT", calibration_windows=3),
        CITRASFM(),
        TabICLv2(),
    ]
)

**6. Genero el forecast y lo guardo**

In [0]:
cv_df = tcf.cross_validation(
    df=df,
    h=12,
    quantiles=[
        0.1, 0.2, 0.3,
        0.5,
        0.7, 0.8, 0.9,
    ]
)
# Se utilizan cuantiles en lugar de niveles de confianza (level)
# para garantizar la compatibilidad con todos los modelos del benchmark,
# incluyendo modelos de machine learning clásico como AutoXGBoost.
#
# Equivalencias:
# Intervalo 40% -> [0.3, 0.7]
# Intervalo 60% -> [0.2, 0.8]
# Intervalo 80% -> [0.1, 0.9]
#
# El cuantil 0.5 corresponde a la mediana (forecast puntual).


In [0]:
#cv_df.to_parquet("/Workspace/Users/anamaria.afonso@practicas.repsol.com/TFM_ANA/Resultados/cv_df.parquet") #para guardar la prediccion y no tener que reejecutarla cada vez que se me apaga el clúster

In [0]:
#leo la predicción guardada para no tener que reejecutarla cada vez que se me apaga el cluster
# Ruta del archivo parquet
#path = "/Workspace/Users/anamaria.afonso@practicas.repsol.com/TFM_ANA/Resultados/cv_df.parquet"

# Cargar el DataFrame
#cv_df = pd.read_parquet(path)

**7. Muestro los resultados**

In [0]:
cv_df.head()

In [0]:
#me representa todos los cuantiles y todos los tamaños de los modelos
tcf.plot(
    df,
    cv_df.drop(columns=["cutoff"])
)

Lo que debo hacer es primero evaluarlos para quedarme solo con el tamaño más adecuado de cada modelo y con su intervalo de confianza y así poder compararlos visualmente

**8. Selección del mejor modelo de cada familia y visualización**

In [0]:
#2. CONFIGURACIÓN
#---------------------------------------------------------
# series son mensuales, por eso la estacionalidad anual es 12.
SEASONALITY = 12

# Niveles que corresponden a los cuantiles calculados.
LEVELS = [40, 60, 80]

# Equivalencias:
#   intervalo 40% = q30-q70
#   intervalo 60% = q20-q80
#   intervalo 80% = q10-q90
LEVEL_TO_QUANTILES = {
    40: (30, 70),
    60: (20, 80),
    80: (10, 90),
}

# Intervalo que se representará en las gráficas.
PLOT_LOWER_QUANTILE = 20
PLOT_CENTRAL_QUANTILE = 50
PLOT_UPPER_QUANTILE = 80

# Número de subgráficos por fila.
N_PLOT_COLUMNS = 2


# ---------------------------------------------------------------------
# 3. COMPROBAR LOS DATAFRAMES DE ENTRADA
# ---------------------------------------------------------------------
required_df_columns = {"unique_id", "ds", "y"}
required_cv_columns = {"unique_id", "ds", "cutoff", "y"}

missing_df_columns = required_df_columns - set(df.columns)
missing_cv_columns = required_cv_columns - set(cv_df.columns)

if missing_df_columns:
    raise ValueError(
        f"Faltan estas columnas en df: {sorted(missing_df_columns)}"
    )

if missing_cv_columns:
    raise ValueError(
        f"Faltan estas columnas en cv_df: "
        f"{sorted(missing_cv_columns)}"
    )

# Trabajamos con copias para no modificar df ni cv_df.
df_work = df.copy()
cv_work = cv_df.copy()

# Asegurar formato de fecha.
df_work["ds"] = pd.to_datetime(df_work["ds"])
cv_work["ds"] = pd.to_datetime(cv_work["ds"])
cv_work["cutoff"] = pd.to_datetime(cv_work["cutoff"])

# Se consideran modelos las columnas que:
#   - no son unique_id, ds, cutoff o y;
#   - no son columnas de cuantiles "-q-";
#   - no son columnas de intervalos "-lo-" o "-hi-".
# ---------------------------------------------------------------------

metadata_columns = {
    "unique_id",
    "ds",
    "cutoff",
    "y",
}

all_models = [
    column
    for column in cv_work.columns
    if column not in metadata_columns
    and "-q-" not in column
    and "-lo-" not in column
    and "-hi-" not in column
]

print(f"Modelos encontrados en cv_df: {len(all_models)}")


# ---------------------------------------------------------------------
# 5. CONSERVAR SOLO MODELOS CON TODOS LOS CUANTILES NECESARIOS
# ---------------------------------------------------------------------
required_quantiles = [10, 20, 30, 50, 70, 80, 90]

models_with_all_quantiles = [
    model
    for model in all_models
    if all(
        f"{model}-q-{quantile}" in cv_work.columns
        for quantile in required_quantiles
    )
]

models_without_all_quantiles = [
    model
    for model in all_models
    if model not in models_with_all_quantiles
]


if models_without_all_quantiles:
    print("\nModelos excluidos por no tener todos los cuantiles:")

    for model in models_without_all_quantiles:
        print(f"  - {model}")

if not models_with_all_quantiles:
    raise ValueError(
        "No hay ningún modelo con todos los cuantiles necesarios."
    )

# ---------------------------------------------------------------------
# 6. CREAR CV_EVAL COMPATIBLE CON UTILSFORECAST
# ---------------------------------------------------------------------
#
# Este dataframe NO debe contener columnas "-q-".
#
# Si las incluyéramos, evaluate() interpretaría, por ejemplo,
# "TimeGPT-q-10" como si fuera otro modelo.
#
# Para cada modelo, cv_eval contendrá únicamente:
#
#   modelo          = q50, predicción central
#   modelo-lo-40    = q30
#   modelo-hi-40    = q70
#   modelo-lo-60    = q20
#   modelo-hi-60    = q80
#   modelo-lo-80    = q10
#   modelo-hi-80    = q90
# ---------------------------------------------------------------------

cv_eval = cv_work[
    ["unique_id", "ds", "cutoff", "y"]
].copy()

# Para evitar avisos de fragmentación de pandas, creamos primero
# todas las columnas nuevas en un diccionario.
forecast_columns = {}

for model in models_with_all_quantiles:

    # Predicción puntual: mediana q50.
    forecast_columns[model] = cv_work[f"{model}-q-50"]

    # Intervalos requeridos por evaluate(level=[40, 60, 80]).
    for level, (lower_q, upper_q) in LEVEL_TO_QUANTILES.items():

        forecast_columns[f"{model}-lo-{level}"] = (
            cv_work[f"{model}-q-{lower_q}"]
        )

        forecast_columns[f"{model}-hi-{level}"] = (
            cv_work[f"{model}-q-{upper_q}"]
        )

cv_eval = pd.concat(
    [
        cv_eval,
        pd.DataFrame(
            forecast_columns,
            index=cv_eval.index,
        ),
    ],
    axis=1,
)

# ---------------------------------------------------------------------
# 7. VALIDAR CV_EVAL ANTES DE EVALUAR
# ---------------------------------------------------------------------

# No debe haber ninguna columna "-q-".
quantile_columns_accidentally_included = [
    column
    for column in cv_eval.columns
    if "-q-" in column
]

if quantile_columns_accidentally_included:
    raise ValueError(
        "cv_eval contiene columnas de cuantiles que no deberían estar: "
        f"{quantile_columns_accidentally_included[:10]}"
    )

# Comprobar que existen todos los intervalos.
required_interval_columns = {
    f"{model}-{bound}-{level}"
    for model in models_with_all_quantiles
    for level in LEVELS
    for bound in ["lo", "hi"]
}

missing_interval_columns = (
    required_interval_columns - set(cv_eval.columns)
)

if missing_interval_columns:
    raise ValueError(
        "Faltan columnas de intervalos: "
        f"{sorted(missing_interval_columns)[:20]}"
    )

print(
    f"cv_eval preparado correctamente con "
    f"{len(models_with_all_quantiles)} modelos."
)

print(
    "¿Contiene columnas '-q-'?:",
    any("-q-" in column for column in cv_eval.columns),
)



In [0]:
# 8. CONSTRUIR TRAIN_DF
# ---------------------------------------------------------------------
#
# MASE, MSSE, RMSSE y SPIS necesitan el histórico de entrenamiento.
#
# Se utiliza el primer cutoff de cada serie para impedir que entren
# observaciones correspondientes al periodo de validación.
# ---------------------------------------------------------------------

first_cutoff_by_series = (
    cv_eval
    .groupby("unique_id", as_index=False)["cutoff"]
    .min()
    .rename(columns={"cutoff": "first_cutoff"})
)

train_df = (
    df_work
    .merge(
        first_cutoff_by_series,
        on="unique_id",
        how="inner",
    )
)

train_df = (
    train_df.loc[
        train_df["ds"] <= train_df["first_cutoff"],
        ["unique_id", "ds", "y"],
    ]
    .copy()
)

# ---------------------------------------------------------------------
# 9. CALCULAR TODAS LAS MÉTRICAS
# ---------------------------------------------------------------------
#
# Importante:
#   - cv_eval sí tiene columnas lo/hi.
#   - cv_eval no tiene columnas q.
#   - level informa a evaluate de los niveles existentes.
# ---------------------------------------------------------------------

eval_df = evaluate(
    cv_eval.drop(columns=["cutoff"]),
    train_df=train_df,
    level=LEVELS,
    metrics=[
        # Errores absolutos y cuadráticos
        mae,
        rmse,

        # Errores porcentuales
        smape,

        # Errores escalados
        partial(
            mase,
            seasonality=SEASONALITY,
        ),

        # Métricas de inventario
        pis,
        spis,

        # Métricas probabilísticas
        coverage,
        calibration,
        winkler_score,
        scaled_crps,
    ],
)


In [0]:
# 10. CREAR TABLA DE RESULTADOS
# ---------------------------------------------------------------------
#
# evaluate() devuelve resultados por serie y métrica.
# Se calcula la media entre todas las series.
#
# Las filas serán modelos y las columnas serán métricas.
# ---------------------------------------------------------------------

scores = (
    eval_df
    .groupby("metric")
    .mean(numeric_only=True)
    .T
)
# 1. Limpiamos el nombre del eje de las columnas
scores.columns.name = None
# 2. Le damos nombre al índice para saber que son los modelos
scores.index.name = "model"
# Conservar exclusivamente los modelos reales.
scores = scores.loc[
    scores.index.intersection(models_with_all_quantiles)
].copy()
# 3. (Opcional pero recomendado) Convertimos el índice en una columna normal
# para que al hacer display() o guardarlo se vea perfecta.
scores = scores.reset_index()
if scores.empty:
    raise ValueError(
        "evaluate() no ha producido resultados para ningún modelo."
    )


# ---------------------------------------------------------------------
# 11. PREPARAR LAS MÉTRICAS PARA EL RANKING
# ---------------------------------------------------------------------

ranking_values = scores.copy()

metrics_excluded_from_ranking = []

for metric_name in list(ranking_values.columns):

    metric_text = str(metric_name).lower()

    # Sesgo, error acumulado y calibración:
    # cuanto más cerca de cero, mejor.
    if (
        metric_text == "bias"
        or metric_text == "cfe"
        or "calibration" in metric_text
    ):
        ranking_values[metric_name] = (
            ranking_values[metric_name].abs()
        )

    # Coverage:
    # cuanto más cerca de la cobertura nominal, mejor.
    elif "coverage" in metric_text:

        nominal_level = None

        for level in LEVELS:
            if str(level) in metric_text:
                nominal_level = level
                break

        if nominal_level is None:
            # Si el nombre de la métrica no indica si corresponde
            # al 40%, 60% u 80%, no puede incluirse correctamente
            # en el ranking.
            #
            # Se mantiene en scores para inspeccionarla.
            metrics_excluded_from_ranking.append(metric_name)

        else:
            coverage_values = ranking_values[metric_name]

            # Detectar si coverage usa escala 0-1 o 0-100.
            if coverage_values.abs().max() <= 1.5:
                target_coverage = nominal_level / 100
            else:
                target_coverage = nominal_level

            ranking_values[metric_name] = (
                coverage_values - target_coverage
            ).abs()


# Eliminar únicamente las coberturas cuyo nivel no pueda identificarse.
if metrics_excluded_from_ranking:

    ranking_values = ranking_values.drop(
        columns=metrics_excluded_from_ranking
    )

# ---------------------------------------------------------------------
# 12. TRATAR NAN E INFINITOS
# ---------------------------------------------------------------------
#
# MAPE puede producir infinito cuando y = 0.
#
# Una métrica se usa si tiene al menos un valor finito.
# Cuando un modelo concreto tenga NaN, quedará al final en esa métrica.
# ---------------------------------------------------------------------

ranking_values = ranking_values.replace(
    [np.inf, -np.inf],
    np.nan,
)

valid_ranking_metrics = [
    metric_name
    for metric_name in ranking_values.columns
    if ranking_values[metric_name].notna().any()
]

ranking_values = ranking_values[
    valid_ranking_metrics
].copy()

if ranking_values.empty:
    raise ValueError(
        "No hay métricas válidas para crear el ranking."
    )



In [0]:
# 13. CALCULAR RANKING GLOBAL
# ---------------------------------------------------------------------
#
# Cada modelo recibe una posición dentro de cada métrica:
#
#   1 = mejor
#   2 = segundo
#   3 = tercero
#
# rank_score es la posición media.
#
# Menor rank_score = mejor resultado global.
# Todas las métricas válidas tienen el mismo peso.
# ---------------------------------------------------------------------

ranks = ranking_values.rank(
    axis=0,
    method="average",
    ascending=True,
    na_option="bottom",
)

scores["rank_score"] = ranks.mean(axis=1)

scores = scores.sort_values(
    by="rank_score",
    ascending=True,
)

ruta_excel = '/Workspace/Users/anamaria.afonso@practicas.repsol.com/TFM_ANA/Resultados/scores.xlsx'

# Guardamos en Excel (index=False evita que se cree una columna extra con números)
scores.to_excel(ruta_excel, index=False)



**8.1. Muestro la tabla completa con todos los modelos, las métricas estudiadas y el rank_score**

In [0]:
display(scores.round(3))

In [0]:
# 14. DEFINIR LAS FAMILIAS DE MODELOS
# ---------------------------------------------------------------------
#
# Se agrupan solo diferentes versiones o tamaños de la misma familia.
#
# Los modelos clásicos se mantienen separados porque AutoARIMA,
# AutoETS, AutoXGBoost, Random Forest, Ridge, etc. son métodos
# diferentes, no tamaños del mismo modelo.
# ---------------------------------------------------------------------

families = {
    # Baseline
    "SeasonalNaive": [
        "SeasonalNaive",
    ],

    # Modelos estadísticos
    "AutoARIMA": [
        "AutoARIMA",
    ],
    "AutoETS": [
        "AutoETS",
    ],

    # Machine learning clásico
    "AutoXGBoost": [
        "AutoXGBoost",
    ],
    "AutoLGBM": [
        "AutoLGBM",
    ],
    "AutoRandomForest": [
        "AutoRandomForest",
    ],
    "AutoLinearRegression": [
        "AutoLinearRegression",
    ],
    "AutoRidge": [
        "AutoRidge",
    ],
    "AutoLasso": [
        "AutoLasso",
    ],

    # TabPFN
    "TabPFN": [
        "TabPFN-2",
        "TabPFN-3",
    ],

    # Chronos
    "Chronos": [
        "amazon/chronos-2",
        "amazon/chronos-bolt-tiny",
        "amazon/chronos-bolt-mini",
        "amazon/chronos-bolt-small",
        "amazon/chronos-bolt-base",
        "amazon/chronos-t5-tiny",
        "amazon/chronos-t5-mini",
        "amazon/chronos-t5-small",
        "amazon/chronos-t5-base",
        "amazon/chronos-t5-large",
    ],

    #TimeMoE
    "TimeMoE": ["TimeMoE-50M"],

    # TimeGPT
    "TimeGPT": [
        "TimeGPT",
    ],

    # Timestral
    "Timestral": [
        "Timestral-5M",
        "Timestral-28M",
        "Timestral-123M",
        "Timestral-209M",
    ],

    # TimesFM
    "TimesFM": [
        "TimesFM-1.0",
        "TimesFM-2.0",
        "TimesFM-2.5",
        "TimesFM-3",
    ],

    # Toto
    "Toto": [
        "Toto-1-151M",
        "Toto-2-4M",
        "Toto-2-22M",
    ],

    # FlowState
    "FlowState": [
        "FlowState",
    ],

    # T0
    "T0": [
        "t0-alpha",
        "t0-beta",
        "t0-gamma",
    ],

    # PatchTST / Granite PatchTST
    "PatchTST": [
        "PatchTST-FM-r1",
        "Granite-PatchTST-FM-r1",
        "Granite-PatchTST-FM-r2",
    ],

    # TimeH2O
    "TimeH2O": [
        "timeh2o-v1",
        "timeh2o-v1-tiny",
    ],
}


# ---------------------------------------------------------------------
# 15. AÑADIR MODELOS NO CLASIFICADOS
# ---------------------------------------------------------------------
#
# Si en el futuro añades otro modelo, no se descartará.
# Se incorporará automáticamente como familia individual.
# ---------------------------------------------------------------------

classified_models = {
    model
    for family_models in families.values()
    for model in family_models
}

unclassified_models = [
    model
    for model in models_with_all_quantiles
    if model not in classified_models
]

for model in unclassified_models:
    families[model] = [model]

if unclassified_models:
    print("\nModelos añadidos como familias individuales:")

    for model in unclassified_models:
        print(f"  - {model}")


In [0]:
# 16. VISTA POR FAMILIA (Sin descartar modelos, solo resumen analítico)
# CREAR RANKING PRINCIPAL
main_ranking = scores.copy()
if main_ranking.index.name != "model" and "model" not in main_ranking.columns:
    main_ranking.index.name = "model"
main_ranking = main_ranking.reset_index()

# Ordenar por el mejor score global
main_ranking = main_ranking.sort_values(by="rank_score", ascending=True)

def get_family(model_name):
    if 'chronos' in model_name.lower(): return 'Chronos'
    if 'timesfm' in model_name.lower(): return 'TimesFM'
    if 'patchtst' in model_name.lower(): return 'PatchTST'
    if 'toto' in model_name.lower(): return 'Toto'
    if 'timestral' in model_name.lower(): return 'Timestral'
    if 'timeh2o' in model_name.lower(): return 'TimeH2O'
    return model_name.split('-')[0]

main_ranking['familia'] = main_ranking['model'].apply(get_family)

family_view = main_ranking.groupby('familia').agg(
    mejor_rank_score=('rank_score', 'min'),
    peor_rank_score=('rank_score', 'max'),
    mejor_smape=('smape', 'min'),
    peor_smape=('smape', 'max'),
    num_variantes=('model', 'count')
).reset_index().sort_values(by='mejor_rank_score', ascending=True)



**8.2. Muestro la tabla resumen por familia**

In [0]:
print("\n--- RESUMEN POR FAMILIA (Impacto de variantes/tamaños) ---")
display(family_view.round(3))

In [0]:
# 3. VISUALIZACIÓN: PREDICCIONES E INCERTIDUMBRE
# =====================================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import numpy as np

PLOT_LOWER_QUANTILE = 20
PLOT_CENTRAL_QUANTILE = 50
PLOT_UPPER_QUANTILE = 80
N_PLOT_COLUMNS = 2

def color_to_rgba(color, alpha=0.15):
    if color.startswith("#"):
        hex_color = color.lstrip("#")
        red = int(hex_color[0:2], 16)
        green = int(hex_color[2:4], 16)
        blue = int(hex_color[4:6], 16)
        return f"rgba({red}, {green}, {blue}, {alpha})"
    if color.startswith("rgb("):
        values = color.replace("rgb(", "").replace(")", "")
        return f"rgba({values}, {alpha})"
    return f"rgba(100, 100, 100, {alpha})"

def prepare_historical(unique_id):
    return (df_work.loc[df_work["unique_id"] == unique_id, ["ds", "y"]]
            .groupby("ds", as_index=False).mean(numeric_only=True).sort_values("ds"))

def prepare_forecast(unique_id, model):
    model_columns = [f"{model}-q-{PLOT_LOWER_QUANTILE}", f"{model}-q-{PLOT_CENTRAL_QUANTILE}", f"{model}-q-{PLOT_UPPER_QUANTILE}"]
    return (cv_work.loc[cv_work["unique_id"] == unique_id, ["ds"] + model_columns]
            .groupby("ds", as_index=False).mean(numeric_only=True).sort_values("ds"))

unique_ids = list(df_work["unique_id"].dropna().unique())

# Seleccionamos los 3 mejores modelos del ranking global para graficar (ajusta este número si quieres ver más)
top_modelos = main_ranking.head(10)

# Asignar colores por familia utilizando plotly.express
color_palette = px.colors.qualitative.Dark24 + px.colors.qualitative.Light24 + px.colors.qualitative.Alphabet
familias_top = top_modelos['familia'].unique()
family_colors = {family: color_palette[index % len(color_palette)] for index, family in enumerate(familias_top)}

In [0]:
# 18. FIGURA INDIVIDUAL PARA LOS MEJORES MODELOS DEL RANKING
# ---------------------------------------------------------------------
individual_figures = {}

for _, row_data in top_modelos.iterrows():
    model = row_data['model']
    family = row_data['familia']
    
    number_of_series = len(unique_ids)
    number_of_rows = int(np.ceil(number_of_series / N_PLOT_COLUMNS))

    figure = make_subplots(
        rows=number_of_rows, cols=N_PLOT_COLUMNS,
        subplot_titles=[str(uid) for uid in unique_ids],
        horizontal_spacing=0.08, vertical_spacing=0.12,
    )

    model_color = family_colors[family]
    interval_color = color_to_rgba(model_color, alpha=0.18)

    for index, unique_id in enumerate(unique_ids):
        row = index // N_PLOT_COLUMNS + 1
        column = index % N_PLOT_COLUMNS + 1

        historical = prepare_historical(unique_id)
        forecast = prepare_forecast(unique_id, model)

        # Histórico
        figure.add_trace(go.Scatter(
            x=historical["ds"], y=historical["y"], mode="lines", name="Valor real",
            legendgroup="actual", showlegend=(index == 0), line=dict(color="black", width=2.3)
        ), row=row, col=column)

        # Límite superior q80
        figure.add_trace(go.Scatter(
            x=forecast["ds"], y=forecast[f"{model}-q-{PLOT_UPPER_QUANTILE}"], mode="lines",
            line=dict(color=model_color, width=0), showlegend=False, hoverinfo="skip", legendgroup=f"{model}-interval"
        ), row=row, col=column)

        # Límite inferior q20 y sombreado
        figure.add_trace(go.Scatter(
            x=forecast["ds"], y=forecast[f"{model}-q-{PLOT_LOWER_QUANTILE}"], mode="lines", fill="tonexty",
            fillcolor=interval_color, line=dict(color=model_color, width=0), name="Intervalo 60% (q20-q80)",
            showlegend=(index == 0), hoverinfo="skip", legendgroup=f"{model}-interval"
        ), row=row, col=column)

        # Predicción central q50
        figure.add_trace(go.Scatter(
            x=forecast["ds"], y=forecast[f"{model}-q-{PLOT_CENTRAL_QUANTILE}"], mode="lines+markers",
            name=f"{family}: {model}", showlegend=(index == 0), legendgroup=f"{model}-forecast",
            line=dict(color=model_color, width=2.5), marker=dict(color=model_color, size=5)
        ), row=row, col=column)

        figure.update_xaxes(title_text="Fecha", row=row, col=column)
        figure.update_yaxes(title_text="Valor", row=row, col=column)

    figure.update_layout(
        title={"text": (f"Modelo Top: {model} (Familia: {family})<br><sup>Predicción q50 e intervalo del 60% (q20-q80)</sup>"), "x": 0.5, "xanchor": "center"},
        template="plotly_white", height=max(550, 400 * number_of_rows), width=1400,
        hovermode="x unified", legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
        margin=dict(l=60, r=40, t=140, b=60)
    )
    individual_figures[model] = figure

In [0]:
# 19. FIGURA CONJUNTA DE LOS MEJORES MODELOS PARA CADA SERIE
# ---------------------------------------------------------------------
comparison_figures = {}

for unique_id in unique_ids:
    figure = go.Figure()
    historical = prepare_historical(unique_id)

    # Histórico real
    figure.add_trace(go.Scatter(
        x=historical["ds"], y=historical["y"], mode="lines", name="Valor real", line=dict(color="black", width=3)
    ))

    for _, row_data in top_modelos.iterrows():
        model = row_data['model']
        family = row_data['familia']
        
        model_color = family_colors[family]
        interval_color = color_to_rgba(model_color, alpha=0.10)
        forecast = prepare_forecast(unique_id, model)

        # Límite superior q80
        figure.add_trace(go.Scatter(
            x=forecast["ds"], y=forecast[f"{model}-q-{PLOT_UPPER_QUANTILE}"], mode="lines",
            line=dict(color=model_color, width=0), showlegend=False, hoverinfo="skip", legendgroup=model
        ))

        # Límite inferior q20 y sombreado
        figure.add_trace(go.Scatter(
            x=forecast["ds"], y=forecast[f"{model}-q-{PLOT_LOWER_QUANTILE}"], mode="lines", fill="tonexty",
            fillcolor=interval_color, line=dict(color=model_color, width=0), showlegend=False, hoverinfo="skip", legendgroup=model
        ))

        # Predicción central q50
        figure.add_trace(go.Scatter(
            x=forecast["ds"], y=forecast[f"{model}-q-{PLOT_CENTRAL_QUANTILE}"], mode="lines+markers",
            name=f"{model}", legendgroup=model, line=dict(color=model_color, width=2.2), marker=dict(color=model_color, size=4)
        ))

        figure.update_layout(
            title={"text": (f"Comparativa Top 10 Modelos: Serie {unique_id}<br><sup>Predicción q50 e intervalo del 60% (q20-q80)</sup>"), "x": 0.5, "xanchor": "center"},
            template="plotly_white", height=800, width=1550, hovermode="x unified",
            xaxis_title="Fecha", yaxis_title="Valor",
            legend=dict(title="Top 10 Global", orientation="v", yanchor="top", y=1, xanchor="left", x=1.01),
            margin=dict(l=70, r=380, t=120, b=70)
        )
        comparison_figures[unique_id] = figure
    figure.show()

In [0]:
# 19. FIGURA CONJUNTA DE LOS MEJORES MODELOS PARA CADA SERIE (Y GUARDADO)
# ---------------------------------------------------------------------
import os

# Creamos una carpeta para guardar las figuras ordenadas
output_dir = '/Workspace/Users/anamaria.afonso@practicas.repsol.com/TFM_ANA/figuras/'
os.makedirs(output_dir, exist_ok=True)

comparison_figures = {}

for unique_id in unique_ids:
    figure = go.Figure()
    historical = prepare_historical(unique_id)

    # Histórico real
    figure.add_trace(go.Scatter(
        x=historical["ds"], y=historical["y"], mode="lines", name="Valor real", line=dict(color="black", width=3)
    ))

    for _, row_data in top_modelos.iterrows():
        model = row_data['model']
        family = row_data['familia']
        
        model_color = family_colors[family]
        interval_color = color_to_rgba(model_color, alpha=0.10)
        forecast = prepare_forecast(unique_id, model)

        # Límite superior q80
        figure.add_trace(go.Scatter(
            x=forecast["ds"], y=forecast[f"{model}-q-{PLOT_UPPER_QUANTILE}"], mode="lines",
            line=dict(color=model_color, width=0), showlegend=False, hoverinfo="skip", legendgroup=model
        ))

        # Límite inferior q20 y sombreado
        figure.add_trace(go.Scatter(
            x=forecast["ds"], y=forecast[f"{model}-q-{PLOT_LOWER_QUANTILE}"], mode="lines", fill="tonexty",
            fillcolor=interval_color, line=dict(color=model_color, width=0), showlegend=False, hoverinfo="skip", legendgroup=model
        ))

        # Predicción central q50
        figure.add_trace(go.Scatter(
            x=forecast["ds"], y=forecast[f"{model}-q-{PLOT_CENTRAL_QUANTILE}"], mode="lines+markers",
            name=f"{model}", legendgroup=model, line=dict(color=model_color, width=2.2), marker=dict(color=model_color, size=4)
        ))

    # ACTUALIZACIÓN DEL LAYOUT (Fuera del bucle de modelos, dentro del bucle de unique_id)
    figure.update_layout(
        title={"text": (f"Comparativa Top 10 Modelos: Serie {unique_id}<br><sup>Predicción q50 e intervalo del 60% (q20-q80)</sup>"), "x": 0.5, "xanchor": "center"},
        template="plotly_white", height=800, width=1550, hovermode="x unified",
        xaxis_title="Fecha", yaxis_title="Valor",
        legend=dict(title="Top 10 Global", orientation="v", yanchor="top", y=1, xanchor="left", x=1.01),
        margin=dict(l=70, r=380, t=120, b=70)
    )
    
    comparison_figures[unique_id] = figure

    figure.show()
